In [22]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.cox import *
from functions.repeated_cv import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis

import jinja2

import time
set_config(display="text")  # displays text representation of estimators

In [23]:
dataset = pd.read_csv("../../data/data_final.csv")

diagnosis_variables = []

with open("../../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [24]:
# Find duplicated patients with regional ln recurrence = 1
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [25]:
survival_dataset = survival_dataset[
    ~((survival_dataset['Survival days'] > 1825) & (survival_dataset['Regional LN recurrence'] == 1))
]

In [26]:
# Prepare data
data_x = survival_dataset[feature_columns]

data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
# Since every duplicated patient has a consistent event indicator (regional ln recurrence) I can use this method
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

In [27]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])

train_index, test_index = next(cv)

In [28]:
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

In [29]:
# Define hyperparameter ranges
alpha_range = np.logspace(-4, -1, 10)
l1_ratio_range = [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]

metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

# Run grid search
results, summary_df, best_params = cox_repeated_grid_search(x_train, y_train, alpha_range, l1_ratio_range, metrics,
                             scale_data=True, scaler_type='standard',
                             binary_features=None, continuous_features=None,
                             n_splits=4, n_repeats=5, base_random_state=42,
                             verbose=True)

COX MODEL REPEATED GRID SEARCH
L1 ratios: [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]
Alphas: 10 values from 1.00e-04 to 1.00e-01
Total hyperparameter combinations: 60
Folds per repetition: 4
Repetitions: 5
Total evaluations: 1200
Scaling: Yes (standard)

[1/60] l1=1.000, alpha=1.00e-04 (elapsed: 0.0min, est. remaining: 0.0min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations

[2/60] l1=1.000, alpha=2.15e-04 (elapsed: 7.9min, est. remaining: 227.7min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations

[3/60] l1=1.000, alpha=4.64e-04 (elapsed: 13.3min, est. remaining: 253.3min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations

... (25 outputs left out)

c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:86: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:86: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want

[54/60] l1=0.001, alpha=1.00e-03 (elapsed: 62.9min, est. remaining: 7.0min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:86: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)


[55/60] l1=0.001, alpha=2.15e-03 (elapsed: 63.4min, est. remaining: 5.8min)
[56/60] l1=0.001, alpha=4.64e-03 (elapsed: 63.5min, est. remaining: 4.5min)
[57/60] l1=0.001, alpha=1.00e-02 (elapsed: 63.7min, est. remaining: 3.4min)
[58/60] l1=0.001, alpha=2.15e-02 (elapsed: 63.8min, est. remaining: 2.2min)
[59/60] l1=0.001, alpha=4.64e-02 (elapsed: 63.9min, est. remaining: 1.1min)
[60/60] l1=0.001, alpha=1.00e-01 (elapsed: 64.1min, est. remaining: 0.0min)

REPEATED GRID SEARCH COMPLETE
Total time: 64.2 minutes

Best parameters for each metric:
--------------------------------------------------------------------------------

c-index:
  l1_ratio: 1.0000
  alpha: 4.6416e-04
  Score: 0.9113 ± 0.0520

c-index-ipcw:
  l1_ratio: 0.5000
  alpha: 4.6416e-04
  Score: 0.9033 ± 0.0627

int-brier-score:
  l1_ratio: 0.0010
  alpha: 2.1544e-02
  Score: 0.0087 ± 0.0010




In [31]:
summary_df.to_csv('cox_sub_5.csv', index=False)

In [32]:
# ============================================================================
# CLEAN COX DATA: Remove rows with missing fold values (gaps)
# ============================================================================
def clean_cox_data(df):
    """
    Remove rows that have any NaN values in the c-index or c-index-ipcw fold columns.
    This ensures we only keep configurations where all folds computed successfully.
    """
    # Get all fold columns for c-index (normal)
    cindex_fold_cols = [col for col in df.columns if col.startswith('c-index_repeat_') and '_fold_' in col]
    
    # Get all fold columns for c-index-ipcw
    ipcw_fold_cols = [col for col in df.columns if col.startswith('c-index-ipcw_repeat_') and '_fold_' in col]
    
    # Combine all fold columns
    all_fold_cols = cindex_fold_cols + ipcw_fold_cols
    
    # Count rows before cleaning
    rows_before = len(df)
    
    # Keep only rows where ALL fold columns have valid (non-NaN) values
    df_clean = df.dropna(subset=all_fold_cols, how='any')
    
    rows_after = len(df_clean)
    rows_removed = rows_before - rows_after
    
    print(f"  Checked {len(cindex_fold_cols)} c-index fold columns and {len(ipcw_fold_cols)} c-index-ipcw fold columns")
    print(f"  Removed {rows_removed} rows with missing fold values ({rows_before} -> {rows_after})")
    
    return df_clean

print("\nCleaning Cox Full data...")
cox_full_clean = clean_cox_data(summary_df)

# Save cleaned data to CSV
print("\nSaving cleaned data to CSV...")
cox_full_clean.to_csv('cox_full_cleaned_5.csv', index=False)

print(f"\nSaved: cox_full_cleaned_5.csv ({cox_full_clean.shape[0]} rows)")
# Update the original variables with cleaned data
cox_full = cox_full_clean


Cleaning Cox Full data...
  Checked 20 c-index fold columns and 20 c-index-ipcw fold columns
  Removed 20 rows with missing fold values (60 -> 40)

Saving cleaned data to CSV...

Saved: cox_full_cleaned_5.csv (40 rows)


In [33]:
# Function to find hyperparameter combination with lowest coefficient of variation among top performers
def find_lowest_cv_config(df, metric='c-index', top_n=20):
    """
    Among the top N configurations by mean performance, find the one with the lowest 
    coefficient of variation (CV = std/mean).
    
    Parameters:
    -----------
    df : DataFrame
        Results dataframe with mean and std columns
    metric : str
        Metric name to evaluate (default: 'c-index')
    top_n : int
        Number of top configurations to consider (default: 20)
    
    Returns:
    --------
    best_idx : int
        Index of the row with lowest CV among top N
    best_cv : float
        Coefficient of variation of the best configuration
    best_mean : float
        Mean value of the best configuration
    best_std : float
        Std value of the best configuration
    best_rank : int
        Rank of this configuration by mean (1 = highest mean)
    """
    mean_col = f'{metric}_mean'
    std_col = f'{metric}_std'
    
    # Sort by mean (descending) and take top N
    top_configs = df.nlargest(top_n, mean_col)
    
    # Calculate coefficient of variation for top N configs
    cv_values = top_configs[std_col] / top_configs[mean_col]
    
    # Find the index with the minimum CV among top N
    best_idx = cv_values.idxmin()
    
    # Find the rank by mean in the original dataframe
    df_sorted = df.sort_values(mean_col, ascending=False).reset_index(drop=False)
    best_rank = df_sorted[df_sorted['index' if 'index' in df_sorted.columns else df_sorted.columns[0]] == best_idx].index[0] + 1
    
    return (best_idx, 
            cv_values[best_idx], 
            df.loc[best_idx, mean_col], 
            df.loc[best_idx, std_col],
            best_rank)

idx, cv, mean, std, rank = find_lowest_cv_config(cox_full_clean, top_n=20)

In [34]:
print(idx, cv, mean, std, rank)

44 0.07833823776622441 0.8694375836179573 0.06811020814835515 12


In [7]:
# Define hyperparameter ranges
alpha_range = np.logspace(-4, -1, 10)
l1_ratio_range = [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]

metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

# Run grid search
results, summary_df, best_params = cox_repeated_grid_search(x_train, y_train, alpha_range, l1_ratio_range, metrics,
                             scaler_class='standard',
                             binary_features=None, continuous_features=None,
                             n_splits=4, n_repeats=5, base_random_state=42,
                             verbose=True)

COX MODEL REPEATED GRID SEARCH
L1 ratios: [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]
Alphas: 10 values from 1.00e-04 to 1.00e-01
Total hyperparameter combinations: 60
Folds per repetition: 4
Repetitions: 5
Total evaluations: 1200
Scaler: standard

[1/60] l1=1.000, alpha=1.00e-04 (elapsed: 0.0min, est. remaining: 0.0min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations

[2/60] l1=1.000, alpha=2.15e-04 (elapsed: 4.9min, est. remaining: 142.5min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations

[3/60] l1=1.000, alpha=4.64e-04 (elapsed: 8.1min, est. remaining: 153.5min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:69: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:80: RuntimeWarning: invalid value encountered in scalar subtract
  value -= risk_score[k : (k + d)].sum()
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations

... (27 outputs left out)

c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:86: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)
c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: O

[54/60] l1=0.001, alpha=1.00e-03 (elapsed: 32.9min, est. remaining: 3.7min)


c:\Users\Joao Goncalves\OneDrive\University\IST\Tese\thesis_2\functions\cox.py:46: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  model.fit(X_train, y_train)
c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:86: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)


[55/60] l1=0.001, alpha=2.15e-03 (elapsed: 33.4min, est. remaining: 3.0min)
[56/60] l1=0.001, alpha=4.64e-03 (elapsed: 33.6min, est. remaining: 2.4min)
[57/60] l1=0.001, alpha=1.00e-02 (elapsed: 33.7min, est. remaining: 1.8min)
[58/60] l1=0.001, alpha=2.15e-02 (elapsed: 33.9min, est. remaining: 1.2min)
[59/60] l1=0.001, alpha=4.64e-02 (elapsed: 34.0min, est. remaining: 0.6min)
[60/60] l1=0.001, alpha=1.00e-01 (elapsed: 34.1min, est. remaining: 0.0min)

REPEATED GRID SEARCH COMPLETE
Total time: 34.3 minutes

Best parameters for each metric:
--------------------------------------------------------------------------------

c-index:
  l1_ratio: 1.0000
  alpha: 1.0000e-04
  Score: 0.9320 ± nan

c-index-ipcw:
  l1_ratio: 1.0000
  alpha: 1.0000e-04
  Score: 0.9948 ± nan

int-brier-score:
  l1_ratio: 0.1000
  alpha: 2.1544e-02
  Score: 0.0093 ± 0.0015




In [ ]:
summary_df.to_csv('../../results/repeated_full/cox_full.csv', index=False)

: 